# Autonomous Visual AI Agent for Precision Agriculture: SAGE 15-Class Training Pipeline

Fine-Tuning **Qwen2.5-VL-3B-Instruct** using **QLoRA / LoRA (4-bit NF4)** on the SAGE Agricultural Dataset for 15 Verified Indian-Relevant Crop–Disease Classes across 10 Crops.

This notebook runs the complete production pipeline with reproducible metadata sampling, group-level leak prevention, atomic checkpointing, and independent test evaluation.

### Cell 1: Environment Diagnostics
Identify Python version, operating platform, runtime kernel type, and process environment.

In [ ]:
import sys
import os
import platform
from datetime import datetime, timezone

print(f"Timestamp (UTC): {datetime.now(timezone.utc).isoformat()}")
print(f"Python Version: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Working Directory: {os.getcwd()}")
is_kaggle = os.path.exists('/kaggle') or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
print(f"Kaggle Environment Detected: {is_kaggle}")

### Cell 2: GPU Detection and VRAM Reporting
Verify GPU availability, compute capability, VRAM size, and bfloat16 hardware support.

In [ ]:
import torch

cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")
if not cuda_available:
    raise RuntimeError("No GPU detected! In Kaggle, select Settings -> Accelerator -> GPU T4 / P100 / L4.")

device_count = torch.cuda.device_count()
print(f"GPU Device Count: {device_count}")
for i in range(device_count):
    props = torch.cuda.get_device_properties(i)
    vram_gb = round(props.total_memory / (1024**3), 2)
    print(f"  [Device {i}] {props.name} | VRAM: {vram_gb} GB | Compute Capability: {props.major}.{props.minor}")

bf16_supported = hasattr(torch.cuda, 'is_bf16_supported') and torch.cuda.is_bf16_supported()
print(f"Native BFloat16 Supported: {bf16_supported}")
recommended_dtype = torch.bfloat16 if bf16_supported else torch.float16
print(f"Selected Compute Precision: {recommended_dtype}")

### Cell 3: Disk and RAM Availability
Audit system memory and temporary scratch disk storage.

In [ ]:
import shutil
import psutil

vm = psutil.virtual_memory()
total_ram_gb = round(vm.total / (1024**3), 2)
avail_ram_gb = round(vm.available / (1024**3), 2)
print(f"RAM: {avail_ram_gb} GB available / {total_ram_gb} GB total")

target_dir = '/kaggle/working' if is_kaggle else '.'
total_disk, used_disk, free_disk = shutil.disk_usage(target_dir)
free_disk_gb = round(free_disk / (1024**3), 2)
total_disk_gb = round(total_disk / (1024**3), 2)
print(f"Disk Space ({target_dir}): {free_disk_gb} GB free / {total_disk_gb} GB total")

if free_disk_gb < 5.0:
    print("WARNING: Disk space is below 5 GB. Checkpoints and cache should be purged periodically.")

### Cell 4: Install or Verify Compatible Dependencies
Ensure tested versions of PyTorch, Transformers, PEFT, BitsAndBytes, and PyArrow are installed.

In [ ]:
# Install required dependencies if in Kaggle or fresh virtual env
!pip install -q 'transformers>=4.45.0' 'peft>=0.10.0' 'accelerate>=0.28.0' 'bitsandbytes>=0.43.0' 'pyarrow>=14.0.0' 'datasets>=2.18.0' 'huggingface-hub>=0.23.0' 'scikit-learn' 'pyyaml' 'tqdm'
print("Dependencies successfully verified!")

### Cell 5: Clone or Pull the Git Repository
Clone the project repository or link source code.

In [ ]:
# Configure Git repository URL
REPO_URL = os.environ.get('GIT_REPO_URL', '')
BRANCH = os.environ.get('GIT_BRANCH', 'main')

if is_kaggle and REPO_URL:
    if not os.path.exists('sage-agri-agent'):
        !git clone -b {BRANCH} {REPO_URL} sage-agri-agent
    %cd sage-agri-agent

import sys
if 'src' not in sys.path:
    sys.path.insert(0, os.path.abspath('src'))
print("Python path updated with 'src' directory.")

### Cell 6: Print Current Git Commit
Auditing Git commit hash for exact experiment reproducibility.

In [ ]:
import subprocess
try:
    commit = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode('utf-8').strip()
    branch = subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']).decode('utf-8').strip()
    print(f"Git Commit: {commit} (Branch: {branch})")
except Exception as e:
    print(f"Running from local files without Git metadata: {e}")

### Cell 7: Load YAML Configuration Files
Load typed pipeline configuration for dataset, classes, training, and evaluation.

In [ ]:
from sage.config import load_classes_config, load_dataset_config, load_training_config, load_evaluation_config

classes_cfg = load_classes_config('configs/classes.yaml')
dataset_cfg = load_dataset_config('configs/dataset.yaml')
train_cfg = load_training_config('configs/training.yaml')
eval_cfg = load_evaluation_config('configs/evaluation.yaml')

print(f"Approved Target Classes: {classes_cfg.target_classes_count}")
print(f"Target Train Images Per Class: {dataset_cfg.target_train_images_per_class}")
print(f"Training Epochs: {train_cfg.epochs} | Micro-Batch: {train_cfg.micro_batch_size} | Grad Accum: {train_cfg.gradient_accumulation_steps}")

### Cell 8: Locate Dataset Source and Verify Access
Locate local metadata Parquet or download SAGE dataset metadata from Hugging Face.

In [ ]:
import os
import urllib.request

metadata_file = 'sage_metadata.parquet'
if not os.path.exists(metadata_file):
    print("Downloading SAGE metadata (~3.7 MB) from Hugging Face...")
    url = 'https://huggingface.co/datasets/tirtho149/SAGE/resolve/main/sage_metadata.parquet'
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as resp, open(metadata_file, 'wb') as f:
        f.write(resp.read())
    print("Downloaded sage_metadata.parquet successfully!")
else:
    print(f"Found existing {metadata_file} ({round(os.path.getsize(metadata_file)/(1024**2), 2)} MB)")

### Cell 9: Persistent Storage and Artifact Directories
Initialize directory structure for metadata, logs, checkpoints, predictions, and reports.

In [ ]:
for d in ['artifacts/metadata', 'artifacts/logs', 'artifacts/checkpoints', 'artifacts/predictions', 'artifacts/reports', 'artifacts/cache']:
    os.makedirs(d, exist_ok=True)
print("Persistent artifact directories ready.")

### Cell 10: Run Metadata Audit
Execute schema validation and inventory scanning.

In [ ]:
import pandas as pd
from sage.data.inventory import audit_dataset_inventory

df_meta = pd.read_parquet(metadata_file)
print(f"Enumerated raw records: {len(df_meta):,}")
schema_art, inv_art = audit_dataset_inventory(df_meta, classes_cfg, dataset_cfg)
print(f"Records with valid crop & disease: {inv_art['records_with_valid_crop_and_disease']:,}")
print("Saved dataset_schema.json and dataset_inventory.json.")

### Cell 11: Display Class Registry and Verified Counts
Display candidate and verified class representation in tabular format.

In [ ]:
counts_summary = []
for c_name, cnt in sorted(inv_art['candidate_class_record_counts'].items(), key=lambda x: x[1], reverse=True):
    counts_summary.append({'Class Name': c_name, 'Available Records': cnt})
counts_df = pd.DataFrame(counts_summary)
display(counts_df)

### Cell 12: Build and Validate Sample Manifest
Execute balanced per-class sampling targeting 800 train images per class.

In [ ]:
from sage.data.manifest import build_sample_manifest, validate_sample_manifest

manifest_df, split_summary = build_sample_manifest(df_meta, classes_cfg, dataset_cfg)
print(f"Total selected samples in manifest: {len(manifest_df):,}")

### Cell 13: Print Train / Validation / Test Counts
Display exact split distribution.

In [ ]:
print("Split distribution:")
for split_name, count in split_summary['split_counts'].items():
    print(f"  - {split_name.upper()}: {count:,} ({count / len(manifest_df):.1%})")

per_class_table = manifest_df.groupby(['class_name', 'split']).size().unstack(fill_value=0)
display(per_class_table)

### Cell 14: Run Data-Integrity and Split-Leakage Checks
Confirm zero duplicate group leakage across splits.

In [ ]:
v_res = validate_sample_manifest(dataset_cfg.manifest_path)
if not v_res['valid']:
    raise AssertionError(f"Manifest integrity check failed: {v_res['issues']}")
print(f"Manifest Validation Status: PASSED (Zero group leakage detected!)")

### Cell 15: Load Qwen2.5-VL Model and Processor
Load model with 4-bit NF4 quantization and gradient checkpointing.

In [ ]:
from sage.models.load_model import load_qwen_vl_model

model, processor = load_qwen_vl_model(train_cfg, is_training=True)
print(f"Model {train_cfg.model_id} and processor loaded successfully!")

### Cell 16: Configure Quantization and Attach PEFT LoRA Adapters
Attach trainable LoRA adapters and audit parameter counts.

In [ ]:
from sage.models.lora import apply_lora_to_model

model, lora_stats = apply_lora_to_model(model, train_cfg)
print(f"Trainable LoRA Parameters: {lora_stats['trainable_params']:,} ({lora_stats['trainable_percentage']} %)")

### Cell 17: Run Preflight Multimodal Smoke Test
Verify forward and backward passes with real multimodal batch before launching full training.

In [ ]:
import time
from sage.training.train import create_multimodal_batch
from sage.data.streaming import SAGEImageDataset

test_ds = SAGEImageDataset(manifest_df, split='train')
batch_items = [test_ds[0], test_ds[1]]
batch_inputs = create_multimodal_batch(batch_items, processor, train_cfg.instruction)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_inputs = {k: v.to(device) for k, v in batch_inputs.items() if isinstance(v, torch.Tensor)}

t0 = time.time()
out = model(**batch_inputs)
out.loss.backward()
model.zero_grad()
print(f"Preflight forward/backward pass succeeded in {time.time() - t0:.3f}s! Initial loss: {out.loss.item():.4f}")

### Cell 18: Execute Two Complete Training Epochs
Train for 2 complete epochs with visible progress bar, live metrics, and atomic checkpointing.

In [ ]:
from sage.training.train import run_training_pipeline

train_results = run_training_pipeline(
    manifest_df=manifest_df,
    classes_config=classes_cfg,
    dataset_config=dataset_cfg,
    training_config=train_cfg
)
print(f"Training Complete! Final adapter: {train_results['final_adapter_dir']}")

### Cell 19: Evaluate on Validation Split
Compute validation loss and metrics.

In [ ]:
from sage.evaluation.evaluate import run_evaluation_pipeline

val_metrics = run_evaluation_pipeline(
    model=model,
    processor=processor,
    manifest_df=manifest_df,
    classes_config=classes_cfg,
    eval_config=eval_cfg,
    split='val'
)
print(f"Validation Exact Match Accuracy: {val_metrics.get('exact_match_accuracy', 0.0):.4f}")

### Cell 20: Final Independent Test Evaluation
Run final evaluation on held-out test split, generating confusion matrix and reports.

In [ ]:
test_metrics = run_evaluation_pipeline(
    model=model,
    processor=processor,
    manifest_df=manifest_df,
    classes_config=classes_cfg,
    eval_config=eval_cfg,
    split='test'
)
print(f"Test Set Exact Match Accuracy: {test_metrics.get('exact_match_accuracy', 0.0):.4f}")
print(f"Test Set Macro F1: {test_metrics.get('macro_f1', 0.0):.4f}")

### Cell 21: Save Results and Checkpoints
Verify exported artifacts and final report files.

In [ ]:
!ls -lh artifacts/reports
!ls -lh artifacts/checkpoints

### Cell 22: Export Reports and Package Run Archive
Compress trained adapter and reports into a downloadable archive.

In [ ]:
!python scripts/export_results.py --output-tar artifacts/sage_15class_run.tar.gz
print("All deliverables packaged. You can now download 'artifacts/sage_15class_run.tar.gz' from Kaggle output!")